# Agente RAG Especializado com LangChain e Anthropic 🤖📖

Este notebook ensina a construir um sistema de **RAG (Retrieval-Augmented Generation)**, ou Geração Aumentada por Recuperação, utilizando a biblioteca LangChain e o modelo Claude (Anthropic). Criaremos um assistente virtual especializado capaz de responder perguntas técnicas sobre um modelo específico de nobreak (FXP-2000 / FCP-2000) baseando-se estritamente em seu manual PDF e mantendo o histórico da conversa.

## Objetivos de Aprendizagem
* Entender o conceito de **RAG** e como ele resolve o problema de alucinação e falta de conhecimento específico de grandes modelos de linguagem (LLMs).
* Carregar e processar documentos PDF estruturados de forma programática.
* Criar representações vetoriais (**embeddings**) de textos e armazená-las em um banco de vetores nativo em memória (**FAISS**).
* Construir uma cadeia de conversação com controle de estado e memória para manter o contexto de interações passadas.

---

In [ ]:
!pip install -U langchain langchain-classic langchain-community langchain-core langchain-anthropic langchain-huggingface sentence-transformers faiss-cpu pypdf

## Passo 1: Instalação das Dependências 📦

### O que este passo faz?
Instala no ambiente do Google Colab todas as bibliotecas externas necessárias para construir o nosso pipeline de RAG.

### Por que ele é necessário?
O Python por si só não vem com ferramentas nativas para interagir com modelos da Anthropic, fatiar PDFs, gerar embeddings ou processar bancos vetoriais. Usamos as seguintes bibliotecas:
* **langchain** (e suas variações `langchain-core`, `langchain-community`, `langchain-classic`): O ecossistema/framework principal que conecta nossos dados aos modelos de IA.
* **langchain-anthropic**: Integração oficial para usarmos os modelos de IA da Anthropic (como a família Claude).
* **langchain-huggingface**: Conecta o LangChain aos modelos do Hugging Face, que usaremos para converter texto em números (embeddings).
* **sentence-transformers**: O motor matemático que gera os embeddings de texto.
* **faiss-cpu**: Uma biblioteca de busca vetorial extremamente rápida desenvolvida pelo Facebook (Meta).
* **pypdf**: Usada para ler e extrair texto de arquivos PDF.

💡 **Para pensar:** Instalar pacotes com `!pip install` diretamente na célula funciona perfeitamente no Colab, mas em ambientes de produção reais é mais indicado gerenciar dependências de forma estrita em um arquivo `requirements.txt` ou utilizando gerenciadores como `Poetry` ou `Pipenv` para evitar conflitos de versão inesperados.

In [ ]:
import os
from google.colab import userdata
from langchain_anthropic import ChatAnthropic
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import ConversationalRetrievalChain

## Passo 2: Importação de Módulos e Setup do Ambiente 🛠️

### O que este passo faz?
Importa as classes e funções específicas de cada biblioteca instalada que serão usadas ao longo da lógica do nosso sistema.

### Por que ele é necessário?
Em vez de carregar bibliotecas inteiras na memória, importamos apenas as ferramentas que nos interessam (como o leitor de PDF `PyPDFLoader`, o banco de dados `FAISS`, etc.). Isso deixa o código mais limpo e eficiente.

### Como interpretar?
Se a execução terminar sem erros (ícone de check verde), significa que todas as dependências foram corretamente instaladas e os caminhos internos do Python estão prontos para uso.

In [ ]:
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Passo 3: Configuração da Chave de API 🔑

### O que este passo faz?
Configura a variável de ambiente `ANTHROPIC_API_KEY` buscando o valor salvo de forma segura nos segredos (Secrets) do seu Google Colab.

### Por que ele é necessário?
Para usar os modelos inteligentes da Anthropic (Claude), precisamos de uma autenticação oficial. Expor chaves de API diretamente no código de forma explícita é uma prática insegura que pode levar ao roubo de créditos. O `userdata.get()` garante que sua chave fique oculta.

💡 **Para pensar:** Certifique-se de que adicionou um segredo com o nome exato `ANTHROPIC_API_KEY` na barra lateral esquerda do Colab (ícone de chave) e ativou o acesso a este notebook antes de rodar o código.

In [ ]:
loader = PyPDFLoader("manual_avancado_nobreak_fxp2000.pdf")
documents = loader.load()

## Passo 4: Carregamento do Documento PDF 📄

### O que este passo faz?
O `PyPDFLoader` localiza o arquivo do manual do nobreak no armazenamento local, lê seu conteúdo página por página e o armazena na memória como uma lista de documentos.

### Por que ele é necessário?
Para que o modelo de linguagem consiga responder perguntas sobre o nobreak, nós precisamos alimentar o pipeline com as informações reais contidas no manual oficial. Sem isso, o modelo não saberia as especificações técnicas exatas do FXP-2000.

### Como interpretar o resultado?
A variável `documents` conterá uma lista de objetos onde cada objeto representa uma página do arquivo, contendo o texto extraído (`page_content`) e metadados como o número da página.

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()

## Passo 5: Geração de Embeddings e Banco de Vetores (FAISS) 🧠🔢

### O que este passo faz?
1. Define o modelo de **Embeddings** (`BAAI/bge-m3`), responsável por traduzir palavras e conceitos textuais em vetores (listas longas de números) que representam seu significado semântico.
2. Alimenta esses vetores no **FAISS** para criar um índice de busca inteligente.
3. Expõe essa estrutura através de um `retriever` (recuperador).

### Por que ele é necessário?
Modelos de IA não realizam buscas por palavras-chave exatas como o comando "Ctrl+F". Eles buscam por **proximidade de significado**. Ao transformar o texto em vetores de alta dimensão, conseguimos perguntar sobre "problemas de energia" e encontrar trechos que falem de "queda de tensão", mesmo sem usar a palavra exata.

### Como interpretar o resultado?
O `retriever` agora funciona como um bibliotecário ágil: quando fizermos uma pergunta técnica, ele varrerá o banco FAISS em milissegundos e extrairá as páginas mais relevantes do manual para enviar ao modelo de linguagem.

In [ ]:
chat = ChatAnthropic(model="claude-sonnet-5-5", max_tokens=1024)

## Passo 6: Inicialização do Modelo de Linguagem (LLM) 💬

### O que este passo faz?
Instancia a classe de comunicação com o modelo `claude-sonnet-5-5` da Anthropic, configurando o limite máximo de tokens por resposta.

### Por que ele é necessário?
O LLM atuará como o "cérebro" do nosso sistema. Ele receberá a pergunta do usuário junto com os trechos relevantes que o nosso `retriever` encontrou no manual e redigirá uma resposta fluida, natural e correta.

💡 **Para pensar:** O modelo selecionado no código está escrito como `claude-sonnet-5-5`. Dependendo da biblioteca instalada, a nomenclatura oficial da Anthropic para a API pode exigir o nome exato do modelo (como `claude-3-5-sonnet-20241022`). Caso ocorra um erro de modelo não encontrado ao executar, verifique este parâmetro.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
Você é um assistente virtual especializado no atendimento ao cliente no nobreak FCP-2000.
Responda claramente perguntas técnicas, funcionalidades, garantia, manutenção, atualização e suporte técnico.
Se a pergunta for irrelevante, responda educadamente recusando a pergunta.

Use o contexto abaixo para responder:
{context}
"""),
    ("human", "{question}"),
])

## Passo 7: Definição do Prompt do Sistema 📝

### O que este passo faz?
Define a personalidade, as regras de comportamento e o formato do input que será enviado ao modelo.

### Por que ele é necessário?
Sem regras claras, o modelo poderia usar seus próprios conhecimentos prévios e inventar informações caso a resposta não estivesse no manual (alucinação). Instruí-lo a responder apenas com o contexto fornecido (`{context}`) e a recusar educadamente perguntas fora do escopo garante que o assistente permaneça focado e confiável.

💡 **Para pensar:** O prompt faz referência ao nobreak "FCP-2000" no texto explicativo, enquanto o nome do PDF carregado anteriormente continha "fxp2000". É importante garantir que a nomenclatura no manual e a instrução estejam alinhadas para evitar confusões semânticas.

In [ ]:
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=chat,
    retriever=retriever,
    return_source_documents=False,
    combine_docs_chain_kwargs={"prompt": prompt},
)

chain_with_history = RunnableWithMessageHistory(
    qa_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

## Passo 8: Criação da Cadeia de Conversação com Histórico 🔗

### O que este passo faz?
1. Cria um dicionário (`store`) para armazenar o histórico de mensagens de cada sessão ativa.
2. Constrói um `ConversationalRetrievalChain`, que combina o LLM, o recuperador de documentos e o prompt personalizado.
3. Encapsula essa cadeia no `RunnableWithMessageHistory` para gerenciar a persistência das mensagens anteriores automaticamente.

### Por que ele é necessário?
Por padrão, APIs de LLM são stateless (não lembram de nada que foi dito anteriormente). Para que o usuário possa interagir naturalmente dizendo coisas como "E como eu resolvo isso?" (fazendo referência ao problema citado na mensagem anterior), o sistema precisa anexar o histórico da conversa a cada nova requisição.

In [ ]:
# Testes
session_id = "sessao1"
while True:
    pergunta = input("Você: ")
    if pergunta.lower() in ["sair", "quit", "exit"]:
        print("vlw, Até logo")
        break
    resposta = chain_with_history.invoke(
        {"question": pergunta},
        config={"configurable": {"session_id": session_id}},
    )
    print("Assistente:", resposta["answer"])

## Passo 9: Interface de Chat Interativa 💻

### O que este passo faz?
Inicia um loop infinito de entrada de dados (`while True`), permitindo que você digite perguntas diretamente no console e receba as respostas refinadas do assistente em tempo real. Digitar "sair", "quit" ou "exit" encerra a aplicação.

---

## Erros Comuns e Dicas ⚠️
* **FileNotFoundError:** Ocorre se o arquivo `manual_avancado_nobreak_fxp2000.pdf` não for carregado no ambiente do Colab antes da execução do passo 4. Certifique-se de fazer o upload do arquivo para a raiz do seu ambiente.
* **API Key inválida ou ausente:** Se o segredo `ANTHROPIC_API_KEY` não estiver preenchido corretamente no gerenciador de Secrets do Colab, o passo 3 gerará um erro de autenticação.
* **Estouro de Limite (Rate Limit):** Chamar o modelo repetidamente ou passar documentos excessivamente grandes pode estourar as cotas gratuitas da sua chave de API.

---

## Resumo Final 🎯
Neste notebook você aprendeu a:
1. Integrar bibliotecas de ponta para processamento de IA generativa.
2. Carregar e indexar um arquivo PDF local usando modelos de representação semântica (Embeddings).
3. Usar um banco de vetores em memória (FAISS) para buscar trechos específicos baseando-se no contexto da pergunta.
4. Configurar um Agente RAG especializado com memória conversacional persistente.

---

## Exercícios Propostos ✏️
1. **Fácil (Configuração):** Altere a mensagem de instrução do sistema (no passo 7) para fazer com que o assistente responda em um tom mais formal e termine todas as interações com a frase "Deseja mais alguma ajuda técnica?".
2. **Médio (Desempenho):** O texto extraído do PDF pode ser muito grande se processado por página inteira. Altere o carregamento para utilizar um fatiador de texto (`RecursiveCharacterTextSplitter` do LangChain) para quebrar o documento em pedaços (chunks) menores de 500 caracteres antes de gerar os embeddings.
3. **Desafiador (Múltiplas Sessões):** Modifique o loop de testes (passo 9) para perguntar opcionalmente ao usuário qual o ID de sessão que ele deseja usar, permitindo alternar entre duas sessões (`sessao1` e `sessao2`) e demonstrando que o histórico de cada uma permanece isolado no dicionário `store`.

---

## Para ir além 🚀
* Pesquise sobre **Chunking Strategies** (Estratégias de Divisão de Texto) para entender como fatiar seus documentos de maneira ideal para buscas vetoriais.
* Estude sobre **Hybrid Search** (Busca Híbrida) que mistura busca semântica por proximidade vetorial com busca tradicional por palavras-chave (BM25).
* Explore alternativas de bancos de vetores persistentes na nuvem como **ChromaDB**, **Pinecone** ou **PGVector**.